## Computing attention weights and context vectors for *all* input tokens

Given an input matrix

$$
\mathbf{X}=\begin{bmatrix}
\mathbf{x}_1^{\!\top}\\
\mathbf{x}_2^{\!\top}\\
\vdots\\
\mathbf{x}_n^{\!\top}
\end{bmatrix}\in\mathbb{R}^{n\times d},
$$

we want every token $\mathbf{x}_i$ to attend to every other token $\mathbf{x}_j$.
The canonical three-step procedure is:

1. **Attention scores**

   $$
   s_{ij}=\mathbf{x}_i\cdot\mathbf{x}_j
   \quad\Longrightarrow\quad
   \mathbf{S}=\mathbf{X}\mathbf{X}^{\!\top}\in\mathbb{R}^{n\times n}.
   $$

2. **Attention weights** (row-wise softmax)

   $$
   \alpha_{ij}=\frac{\exp(s_{ij})}{\sum_{k=1}^{n}\exp(s_{ik})},
   \qquad
   \mathbf{A}=\operatorname{softmax}(\mathbf{S},\text{dim}=-1).
   $$

3. **Context vectors**

   $$
   \mathbf{C}=\mathbf{A}\mathbf{X}\in\mathbb{R}^{n\times d},
   \quad\text{where row }i\text{ is } \mathbf{c}_i=\sum_{j=1}^{n}\alpha_{ij}\mathbf{x}_j.
   $$

### Naïve double loop (illustrative only)

```{code-block} python
import torch

inputs = torch.randn(5, 3)          # (n=5 tokens, d=3 dims) – toy example
n = inputs.size(0)
scores_loop = torch.empty(n, n)

for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        scores_loop[i, j] = torch.dot(x_i, x_j)

print("Attention scores (loop):\n", scores_loop)
```

### Vectorized implementation

```{code-block} python
scores = inputs @ inputs.T                 # same as double loop
weights = torch.softmax(scores, dim=-1)    # normalize each row
contexts = weights @ inputs                # weighted sum

print("Attention scores (matmul):\n", scores)
print("\nAttention weights:\n", weights)
print("\nContext vectors:\n", contexts)
```

```{code-block} python
# Sanity check: rows of `weights` sum to 1
assert torch.allclose(weights.sum(dim=-1), torch.ones(n))
```

### Explanation of the `dim` argument

`torch.softmax(scores, dim=-1)` applies softmax along the **last** dimension
( here, the column dimension of the $n\times n$ score matrix), ensuring that
each row’s weights add to 1. Setting `dim=0` would erroneously normalize down the columns.

```{tip}
In practice, you would add a scaling factor \(1/\sqrt{d}\) before softmax to stabilize gradients  
and you would often derive **Q**, **K**, and **V** matrices via learned linear projections, as in the original Transformer architecture :cite:`vaswani2017`.
```
